# 历史训练笔记本（输出已清理）
保留早期 action_only 操作记录，不是当前复现入口。当前流程见 ../docs/REPRODUCING.md 与 ../COLAB.md。
原始带输出副本保留在本地 src/mini_wam/training/action_only_train.ipynb，未纳入版本。

# Mini-WAM `action_only`：Colab 正式训练

本 Notebook（交互式笔记本）是训练入口，不重复实现模型或训练循环。核心代码位于 `src/mini_wam/`，本文件负责连接 Colab、准备远程环境、验证训练链路并启动种子 0 的正式训练。

执行顺序：GPU（Graphics Processing Unit，图形处理器）检查 → GitHub 登录与代码获取 → 依赖与数据准备 → Google Drive 挂载 → 前置检查 → 100 步冒烟测试 → 50,000 步正式训练。

> 注意：`/content` 是 Colab 临时磁盘，runtime（运行环境）终止后会被清空；正式 checkpoint（检查点）会同步到 Google Drive。开发阶段禁止运行最终测试场景。

## 1. 检查 Python、PyTorch 和 GPU

确认当前 kernel（内核）确实连接到 Colab GPU。CUDA（Compute Unified Device Architecture，统一计算设备架构）必须可用；断言失败时应重新选择 GPU runtime，而不是继续训练。

### 执行说明

按顺序逐格执行并在每个门禁通过后继续。已经启动正式训练时，不要重新运行前面的环境准备单元；修改本地 Notebook 的文字不会改变已经提交给 Colab kernel 的训练进程。

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert sys.version_info >= (3, 12), (
    f"需要 Python 3.12 或更高版本，当前是 {sys.version}"
)
assert torch.cuda.is_available(), (
    "没有检测到 CUDA，请在右上角重新选择 Colab GPU runtime"
)

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)

## 2. 登录私有 GitHub 仓库

`gh` 是 GitHub CLI（Command Line Interface，命令行界面）。该命令通过浏览器的一次性验证码授权，不要把密码或访问令牌写进 Notebook。新 runtime 通常需要重新登录；已经登录时可跳过。

In [ ]:
!printf 'Y\n' | gh auth login \
  --hostname github.com \
  --git-protocol https \
  --web

## 3. 将项目代码克隆到 Colab 临时磁盘

VS Code 中的 Notebook 文件在本机，但单元格在远程 Colab 执行，因此远程机器也必须拥有项目代码。该单元只适用于全新的 runtime；若 `/content/mini-wam` 已存在，不要重复克隆。最后打印短提交哈希，用于记录本次训练对应的代码版本。

In [ ]:
%cd /content

!gh repo clone mcintoshjody788-collab/mini-wam mini-wam

%cd /content/mini-wam

!git rev-parse --short HEAD

### 历史安装尝试（保留作故障记录，当前流程跳过）

这一格最初在旧提交 `c6f4075` 上执行，当时远程仓库尚无 `requirements-colab.txt`，因此输出中保留了失败记录。仓库更新后它与后面的正式安装格重复；继续当前流程时不要重新执行。

In [ ]:
%cd /content/mini-wam

!python -m pip install -q -r requirements-colab.txt
!python -m pip install -q -e . --no-deps

## 4. 让 Git 使用 GitHub 登录并更新代码

`gh auth setup-git` 把刚才的登录凭据交给普通 `git` 使用；随后只允许 fast-forward（快进）更新，避免在 Colab 中意外产生合并提交。当前正式训练基线的提交应为 `b878252`，并且必须存在 `requirements-colab.txt`。

In [ ]:
!gh auth setup-git

%cd /content/mini-wam
!git pull --ff-only
!git rev-parse --short HEAD
!test -f requirements-colab.txt && echo "requirements-colab.txt 已找到"

## 5. 安装 Colab 专用依赖和项目包

第一条命令安装 Colab 所需的第三方依赖；第二条使用 editable mode（可编辑安装模式）注册本项目，并用 `--no-deps` 防止再次改写 Colab 自带的 PyTorch/CUDA 环境。只有出现明确的 `ERROR` 才视为失败。

In [ ]:
%cd /content/mini-wam

!python -m pip install -q -r requirements-colab.txt
!python -m pip install -q -e . --no-deps

## 6. 下载并核验 Push-T 数据集

数据下载到 `/content/mini-wam/data/lerobot/pusht_image`。脚本不仅下载文件，还验证数据身份和完整性；通过标准固定为 206 个 episode（回合）和 25,650 帧。runtime 终止后本地数据会消失，需要重新下载。

In [ ]:
%cd /content/mini-wam

!python scripts/download_dataset.py

## 7. 挂载 Google Drive

授权当前 Colab runtime 访问 Google Drive。训练在 `/content` 的高速临时磁盘中执行，checkpoint、指标和环境记录再增量镜像到 Drive，以兼顾速度和断线恢复能力。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### 验证 Drive 挂载

用目录存在性断言确认 `MyDrive` 可访问。该检查通过只代表挂载成功；下一步的训练前置检查还会验证实际写入能力和剩余空间。

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
assert drive_root.is_dir(), "Google Drive 挂载失败"
print("Google Drive 已挂载：", drive_root)

## 8. 正式训练前置检查

统一检查 CUDA、GPU、数据指纹、185/21 个训练/验证 episode 划分、配置文件和 Drive 写入条件。只有输出中的 `ready` 为 `true` 才能继续，不能通过跳过检查来掩盖环境问题。

In [ ]:
%cd /content/mini-wam

!python scripts/check_training_ready.py \
  --config configs/action_only_seed0.yaml \
  --run-root /content/drive/MyDrive/mini-wam-runs \
  --require-cuda

## 9. 运行 100 步 GPU 冒烟测试

冒烟测试使用独立配置和目录，只验证 GPU 前向/反向传播、验证、checkpoint 保存与 Drive 镜像链路，不作为正式实验结果。它不会污染 `action-only-seed0` 的正式运行目录。

In [ ]:
%cd /content/mini-wam

!python scripts/train_action_only.py \
  --config configs/action_only_smoke.yaml \
  --run-dir /content/mini-wam-work/smoke \
  --mirror-dir /content/drive/MyDrive/mini-wam-runs/smoke \
  --device cuda

### 检查冒烟测试产物

逐项确认 `last.pt`、`best.pt`、训练/验证指标以及环境和归一化记录已经写入 Google Drive。六项全部显示 `✓` 才说明恢复链路完整。CSV（Comma-Separated Values，逗号分隔值）保存指标表，JSON（JavaScript Object Notation，JavaScript 对象表示法）保存结构化元数据。

In [ ]:
from pathlib import Path

mirror = Path("/content/drive/MyDrive/mini-wam-runs/smoke")

required_files = [
    "checkpoints/last.pt",
    "checkpoints/best.pt",
    "train_metrics.csv",
    "val_metrics.csv",
    "environment.json",
    "normalization.json",
]

for relative_path in required_files:
    path = mirror / relative_path
    print(f"{'✓' if path.exists() else '✗'} {relative_path}")

## 10. 启动种子 0 的正式训练

使用完整训练集运行 50,000 步，`batch_size` 为 128。每 1,000 步完整验证并同步 `last.pt`，验证损失最低的模型另存为 `best.pt`，每 5,000 步额外保留编号归档。首次启动使用本格；runtime 终止后的恢复必须改用带 `--resume` 的命令，不能再次从第 0 步启动。

> 训练运行时不要重复执行本格、切换 GPU 或启动第二个训练进程。最终模型选择固定使用 `best.pt`，不得查看最终测试场景来挑选 checkpoint。

In [ ]:
%cd /content/mini-wam

!python scripts/train_action_only.py \
  --config configs/action_only_seed0.yaml \
  --run-dir /content/mini-wam-work/action-only-seed0 \
  --mirror-dir /content/drive/MyDrive/mini-wam-runs/action-only-seed0 \
  --device cuda